# Day 4 — Tiền xử lý dữ liệu PAPI

**Học máy · Nhóm 15**  
Nguyễn Tiến Sơn (24110054), Trần Thị Ngọc Quyên (24110051)

Mỗi dòng là một tỉnh trong một năm. Bài thực hành dựng lại panel PAPI 2011–2024 từ workbook gốc, rồi chuẩn hóa hai chỉ tiêu dùng cho STING. Mở notebook trong thư mục `notebooks/` của dự án và chọn **Restart Kernel and Run All Cells**.

## 1. Đọc workbook PAPI

Bộ 2011–2019 nhận diện tỉnh bằng tên. Bộ 2020–2024 cung cấp mã tỉnh. Dùng hai hàm đọc sẵn của dự án để thống nhất tên cột chỉ tiêu.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler

cwd = Path.cwd()
for root in (cwd, *cwd.parents):
    if (root / "scripts" / "data" / "build_panel.py").is_file():
        break
else:
    raise FileNotFoundError("Hãy mở notebook trong thư mục dự án STING đã giải nén.")
sys.path.insert(0, str(root))

from scripts.data.build_panel import NEW_FILES, OLD_FILES, load_crosswalk, lookup_code, parse_new, parse_old
from scripts.common.papi_pred_common import load_config, load_panel

records = []
for files, read_workbook in ((OLD_FILES, parse_old), (NEW_FILES, parse_new)):
    for year, filename in files.items():
        read_workbook(year, filename, records)

raw = pd.DataFrame(records)
features = load_config()["features"]
print(f"Đã đọc {len(OLD_FILES) + len(NEW_FILES)} workbook, {len(raw)} dòng tỉnh–năm.")
display(raw.groupby("year").size().rename("Số dòng").to_frame().T)

Đã đọc 14 workbook, 882 dòng tỉnh–năm.


year,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
Số dòng,63,63,63,63,63,63,63,63,63,63,63,63,63,63


## 2. Làm sạch

Kiểm tra dòng trùng và khóa tỉnh–năm. Theo quy ước tiền xử lý của bộ PAPI này, điểm 0 trong cột chỉ tiêu được đưa về giá trị thiếu (`NaN`). So sánh số ô thiếu trước và sau khi đổi 0 để thấy ảnh hưởng đến dữ liệu.

In [2]:
keys = ["province_code", "year"]
scores = [column for column in raw if column not in keys + ["province_ascii"]]

clean = raw.drop_duplicates().copy()
print(f"Số dòng trùng hoàn toàn đã loại: {len(raw) - len(clean)}")

ascii_to_code, code_to_province = load_crosswalk()
old_codes = clean["province_ascii"].map(lambda name: lookup_code(name, ascii_to_code))
clean["province_code"] = clean["province_code"].fillna(old_codes)
if clean["province_code"].isna().any():
    raise ValueError("Còn tên tỉnh chưa quy đổi được sang mã tỉnh.")
clean["province_code"] = clean["province_code"].astype(int)
if clean.duplicated(keys).any():
    raise ValueError("Có dòng cùng khóa tỉnh–năm nhưng khác nội dung.")

missing_before = clean[scores].isna().sum()
zeros = clean[scores].eq(0)
clean[scores] = clean[scores].mask(zeros)
missing_after = clean[scores].isna().sum()

checks = pd.DataFrame({
    "Ô bằng 0 trước xử lý": zeros.sum(),
    "Ô thiếu trước xử lý": missing_before,
    "Ô thiếu sau xử lý": missing_after,
})
checks.loc["Toàn bộ cột điểm"] = checks.sum()
display(checks.loc[[*features, "Toàn bộ cột điểm"]])

Số dòng trùng hoàn toàn đã loại: 0


,Ô bằng 0 trước xử lý,Ô thiếu trước xử lý,Ô thiếu sau xử lý
sd_4_2,4,9,13
sd_6_1,2,7,9
Toàn bộ cột điểm,109,4540,4649


## 3. Tích hợp

Nối bảng mã tỉnh để thống nhất tên tiếng Việt và tên ASCII cho cả 14 năm. Đối chiếu panel vừa dựng với CSV mà notebook đồ án dùng.

In [3]:
provinces = pd.DataFrame.from_dict(code_to_province, orient="index")
provinces = provinces.rename_axis("province_code").reset_index()
panel = clean.drop(columns="province_ascii").merge(
    provinces, on="province_code", how="left", validate="many_to_one"
)
if panel[["province_ascii", "province_vi"]].isna().any().any():
    raise ValueError("Còn mã tỉnh chưa nối được với bảng tên chuẩn.")

reference = load_panel()
panel = panel[reference.columns].sort_values(keys).reset_index(drop=True)
reference = reference.sort_values(keys).reset_index(drop=True)
pd.testing.assert_frame_equal(
    panel, reference, check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12
)

rows, columns = panel.shape
print(
    f"Panel khớp CSV đã công bố: {rows} dòng × {columns} cột, "
    f"{panel['province_code'].nunique()} tỉnh."
)
display(panel[[*keys, "province_vi", *features]].head())

Panel khớp CSV đã công bố: 882 dòng × 41 cột, 63 tỉnh.


,province_code,year,province_vi,sd_4_2,sd_6_1
0,1,2011,Hà Nội,1.585172,1.746413
1,1,2012,Hà Nội,1.499400,1.676689
2,1,2013,Hà Nội,1.630049,1.592731
3,1,2014,Hà Nội,1.564298,1.512394
4,1,2015,Hà Nội,1.533680,1.597215


## 4. Biến đổi dữ liệu

Chọn các dòng có đủ hai chỉ tiêu. Ước lượng trung bình và độ lệch chuẩn từ dữ liệu 2011–2023, rồi dùng cùng bộ chuẩn hóa cho năm 2024. Bảng sau kiểm tra trung bình và độ lệch chuẩn của tập huấn luyện trước và sau biến đổi.

In [4]:
complete = panel.dropna(subset=features)
train = complete.loc[complete["year"] <= 2023, features]
holdout = complete.loc[complete["year"] == 2024, features]

scaler = StandardScaler().fit(train)
train_scaled = pd.DataFrame(scaler.transform(train), columns=features, index=train.index)
holdout_scaled = pd.DataFrame(
    scaler.transform(holdout), columns=features, index=holdout.index
)

comparison = pd.DataFrame({
    "Trung bình gốc": train.mean(),
    "Độ lệch chuẩn gốc": train.std(ddof=0),
    "Trung bình sau chuẩn hóa": train_scaled.mean(),
    "Độ lệch chuẩn sau chuẩn hóa": train_scaled.std(ddof=0),
})
np.testing.assert_allclose(train_scaled.mean(), 0.0, atol=1e-12)
np.testing.assert_allclose(train_scaled.std(ddof=0), 1.0, atol=1e-12)
print(
    f"Dòng thiếu ít nhất một chỉ tiêu: {len(panel) - len(complete)}. "
    f"Đủ dữ liệu: {len(train)} dòng 2011–2023 và {len(holdout)} dòng năm 2024."
)
display(comparison.round(6))

extreme = train_scaled.abs().gt(3).any(axis=1)
print(f"Số dòng huấn luyện có ít nhất một |z| > 3: {extreme.sum()}. Giữ lại để phân cụm.")
print("Mẫu đầu ra chuẩn hóa năm 2024:")
display(panel.loc[holdout.index, ["province_vi", "year"]].join(holdout_scaled).head())

Dòng thiếu ít nhất một chỉ tiêu: 13. Đủ dữ liệu: 808 dòng 2011–2023 và 61 dòng năm 2024.


,Trung bình gốc,Độ lệch chuẩn gốc,Trung bình sau chuẩn hóa,Độ lệch chuẩn sau chuẩn hóa
sd_4_2,1.896950,0.182285,-0.0,1.0
sd_6_1,1.905655,0.138933,-0.0,1.0


Số dòng huấn luyện có ít nhất một |z| > 3: 7. Giữ lại để phân cụm.
Mẫu đầu ra chuẩn hóa năm 2024:


,province_vi,year,sd_4_2,sd_6_1
13,Hà Nội,2024,0.185649,-1.579195
27,Hà Giang,2024,1.528925,0.170092
41,Cao Bằng,2024,0.484818,0.391267
55,Bắc Kạn,2024,1.203470,1.224135
69,Tuyên Quang,2024,0.430225,1.082105


## Kết luận

1. Đọc và đồng nhất các cột chỉ tiêu từ hai định dạng workbook.
2. Kiểm tra dòng trùng, quy đổi mã tỉnh và đổi điểm 0 theo quy ước nguồn về `NaN`.
3. Nối bảng tên tỉnh, sắp xếp theo mã tỉnh và năm, rồi đối chiếu panel 882 × 41 với CSV của đồ án.
4. Lọc các dòng có đủ hai chỉ tiêu STING. Fit `StandardScaler` trên 2011–2023 và dùng lại cho 2024. Các dòng có `|z| > 3` được báo để rà soát và vẫn nằm trong dữ liệu.